<a href="https://colab.research.google.com/github/li10637/HanLP/blob/doc-zh/chapter_builders-guide/lazy-init.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

In [ ]:
!pip install d2l==1.0.3


# Lazy Initialization
:label:`sec_lazy_init`

So far, it might seem that we got away
with being sloppy in setting up our networks.
Specifically, we did the following unintuitive things,
which might not seem like they should work:

* We defined the network architectures
  without specifying the input dimensionality.
* We added layers without specifying
  the output dimension of the previous layer.
* We even "initialized" these parameters
  before providing enough information to determine
  how many parameters our models should contain.

You might be surprised that our code runs at all.
After all, there is no way the deep learning framework
could tell what the input dimensionality of a network would be.
The trick here is that the framework *defers initialization*,
waiting until the first time we pass data through the model,
to infer the sizes of each layer on the fly.


Later on, when working with convolutional neural networks,
this technique will become even more convenient
since the input dimensionality
(e.g., the resolution of an image)
will affect the dimensionality
of each subsequent layer.
Hence the ability to set parameters
without the need to know,
at the time of writing the code,
the value of the dimension
can greatly simplify the task of specifying
and subsequently modifying our models.
Next, we go deeper into the mechanics of initialization.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append("/content/drive/MyDrive")
import torch
from torch import nn
from d2l import torch as d2l


Mounted at /content/drive


To begin, let's instantiate an MLP.


In [3]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))

At this point, the network cannot possibly know
the dimensions of the input layer's weights
because the input dimension remains unknown.


Consequently the framework has not yet initialized any parameters.
We confirm by attempting to access the parameters below.


In [5]:
net[0].weight  # 网络不知道输入层的权重，因为输入维度未知，框架也没有初始化任何参数

<UninitializedParameter>

Next let's pass data through the network
to make the framework finally initialize parameters.


In [7]:
X = torch.rand(2, 20) # 通过网络传递数据，最终构建框架初始化参数
net(X)

net[0].weight.shape,net[2].weight.shape

(torch.Size([256, 20]), torch.Size([10, 256]))

As soon as we know the input dimensionality,
20,
the framework can identify the shape of the first layer's weight matrix by plugging in the value of 20.
Having recognized the first layer's shape, the framework proceeds
to the second layer,
and so on through the computational graph
until all shapes are known.
Note that in this case,
only the first layer requires lazy initialization,
but the framework initializes sequentially.
Once all parameter shapes are known,
the framework can finally initialize the parameters.


The following method
passes in dummy inputs
through the network
for a dry run
to infer all parameter shapes
and subsequently initializes the parameters.
It will be used later when default random initializations are not desired.


In [10]:
@d2l.add_to_class(d2l.Module)
def apply_init(self, inputs, init=None): # 该方法将虚拟输入传递到网络中，在默认随机初始化时使用
    self.forward(*inputs)
    if init is not None:
        self.net.apply(init)

In [12]:
# 练习1
net = nn.Sequential(nn.Linear(20,256), nn.ReLU(), nn.LazyLinear(10))
print(net[0].weight,net[2].weight)
X = torch.rand(2, 20)
net(X)

print(net[0].weight.shape,net[2].weight.shape)

Parameter containing:
tensor([[ 0.2046,  0.2119,  0.1155,  ...,  0.0353, -0.1564,  0.1012],
        [ 0.0148, -0.1316,  0.0586,  ..., -0.0354,  0.1099,  0.1362],
        [ 0.1713, -0.0618, -0.1754,  ..., -0.1163,  0.1285,  0.1308],
        ...,
        [ 0.1521,  0.0389, -0.0955,  ...,  0.2138,  0.2011, -0.2069],
        [ 0.0005, -0.1218,  0.1455,  ..., -0.1682,  0.0752, -0.0294],
        [-0.1165,  0.1801,  0.0404,  ...,  0.1583,  0.1232,  0.0556]],
       requires_grad=True) <UninitializedParameter>
torch.Size([256, 20]) torch.Size([10, 256])


In [13]:
# 练习2
net = nn.Sequential(nn.Linear(20,256), nn.ReLU(), nn.LazyLinear(10))
print(net[0].weight,net[2].weight)
X = torch.rand(2, 10)
net(X)

print(net[0].weight.shape,net[2].weight.shape)

Parameter containing:
tensor([[ 0.1006, -0.1996, -0.0978,  ..., -0.2192, -0.1620,  0.0429],
        [-0.1761,  0.1329, -0.1326,  ...,  0.2167,  0.1987, -0.1457],
        [ 0.2000, -0.1557, -0.1976,  ..., -0.1805,  0.1771, -0.2191],
        ...,
        [-0.1854, -0.0390,  0.1461,  ..., -0.2012, -0.0893, -0.1079],
        [ 0.0136,  0.1706, -0.2183,  ...,  0.1488, -0.0740, -0.2041],
        [-0.1825, -0.1433,  0.0662,  ...,  0.0795, -0.0500,  0.1969]],
       requires_grad=True) <UninitializedParameter>


RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x10 and 20x256)

In [21]:
# 练习3
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
net = nn.Sequential(nn.Linear(20,256), nn.ReLU(), nn.LazyLinear(10))
print(net[0].weight,net[2].weight)
X = torch.rand(2, 10)
X_padded = F.pad(X, (0, 10))
print(f"填充后形状: {X_padded.shape}")
net(X_padded)
print(net[0].weight.shape,net[2].weight.shape)

Y = torch.rand(20, 100)
scaler = StandardScaler()
Y_scaled = scaler.fit_transform(Y)
pca = PCA(n_components=20)
Y_pca = pca.fit_transform(Y_scaled)
print(f"降维后形状: {Y_pca.shape}")



Parameter containing:
tensor([[ 0.1832, -0.0540, -0.0519,  ...,  0.0033,  0.1056, -0.0520],
        [-0.0180, -0.0703, -0.0580,  ..., -0.1120,  0.0563,  0.0521],
        [-0.1444, -0.1592,  0.1877,  ...,  0.1153, -0.1156,  0.1373],
        ...,
        [-0.1674, -0.1401, -0.2052,  ...,  0.1399, -0.1965,  0.2231],
        [ 0.0605, -0.1941,  0.1947,  ..., -0.0745,  0.1906,  0.0076],
        [-0.1724, -0.0210, -0.1616,  ..., -0.0019,  0.1517,  0.1486]],
       requires_grad=True) <UninitializedParameter>
填充后形状: torch.Size([2, 20])
torch.Size([256, 20]) torch.Size([10, 256])
降维后形状: (20, 20)


## Summary

Lazy initialization can be convenient, allowing the framework to infer parameter shapes automatically, making it easy to modify architectures and eliminating one common source of errors.
We can pass data through the model to make the framework finally initialize parameters.


## Exercises

1. What happens if you specify the input dimensions to the first layer but not to subsequent layers? Do you get immediate initialization?
1. What happens if you specify mismatching dimensions?
1. What would you need to do if you have input of varying dimensionality? Hint: look at the parameter tying.




*   1、指定了第一层的输入尺寸，没有指定后续层尺寸，会正常运行，第一层会立即初始化，但其它层通融羊刀数据第一次通过模型传递才会初始化。
*   2、会因为矩阵乘法的维度不匹配而报错
*   3、如果输入维度比指定维度小，可用padding填充；反之可考虑用pca降维



[Discussions](https://discuss.d2l.ai/t/8092)
